# 05 — Phase 2 vs Phase 3: sklearn vs Spark MLlib Comparison

**What this notebook does:**
- Trains 3 classifiers (Decision Tree, Random Forest, Naive Bayes) using Spark MLlib
  on the same crime category prediction task as Phase 2
- Compares MLlib results against Phase 2 sklearn metrics
- Produces a side-by-side comparison table

**Serverless constraints applied:**
- `dense_rank()` replaces `StringIndexer` (Py4J whitelist block)
- Small model sizes (numTrees=30, maxDepth=5) to fit within 256MB model cache limit
- `del model` between each fit to stay within 1GB session cache
- No `CrossValidator` (causes cache overflow)

**Prerequisite:** Run `02_silver_layer.ipynb` first.

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/raw_data/sparkml_temp"

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    NaiveBayes
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
import time
import gc

print("Imports OK")

## 1. Load & Prepare Data

In [0]:
df = spark.read.table("silver_mydata")
print(f"Total rows: {df.count():,}")

model_df = (
    df.filter(F.col("Crm_Cd_Desc").isNotNull())
    .select(
        F.col("AREA").cast(DoubleType()),
        F.col("Hour").cast(DoubleType()),
        F.col("Month").cast(DoubleType()),
        F.col("IsWeekend").cast(DoubleType()),
        F.col("Has_Weapon").cast(DoubleType()),
        F.col("Reporting_Delay").cast(DoubleType()),
        F.coalesce(F.col("Premis_Desc"), F.lit("Unknown")).alias("Premis_Desc"),
        F.coalesce(F.col("Vict_Sex_Clean"), F.lit("Unknown")).alias("Vict_Sex"),
        F.coalesce(F.col("Vict_Descent"), F.lit("Unknown")).alias("Vict_Descent"),
        F.col("AREA_NAME"),
        F.col("Crm_Cd_Desc"),
    )
    .fillna(0.0, subset=["AREA", "Hour", "Month", "IsWeekend",
                          "Has_Weapon", "Reporting_Delay"])
)

print(f"Model dataset: {model_df.count():,} rows")
print(f"Target classes: {model_df.select('Crm_Cd_Desc').distinct().count()}")

## 2. Categorical Encoding via SQL (dense_rank)

In [0]:
cat_cols = ["Premis_Desc", "Vict_Sex", "Vict_Descent", "AREA_NAME"]
num_cols = ["AREA", "Hour", "Month", "IsWeekend", "Has_Weapon", "Reporting_Delay"]

encoded_df = model_df
for col_name in cat_cols:
    freq_df = (
        model_df.groupBy(col_name)
        .count()
        .withColumn("_rank", F.dense_rank().over(
            Window.orderBy(F.desc("count"))
        ) - 1)
        .select(
            F.col(col_name).alias(f"_join_{col_name}"),
            F.col("_rank").cast(DoubleType()).alias(f"{col_name}_idx")
        )
    )
    encoded_df = (
        encoded_df
        .join(freq_df,
              encoded_df[col_name] == freq_df[f"_join_{col_name}"],
              "left")
        .drop(f"_join_{col_name}")
    )

label_freq = (
    model_df.groupBy("Crm_Cd_Desc")
    .count()
    .withColumn("_label", F.dense_rank().over(
        Window.orderBy(F.desc("count"))
    ) - 1)
    .select(
        F.col("Crm_Cd_Desc").alias("_join_label"),
        F.col("_label").cast(DoubleType()).alias("label")
    )
)
encoded_df = (
    encoded_df
    .join(label_freq,
          encoded_df["Crm_Cd_Desc"] == label_freq["_join_label"],
          "left")
    .drop("_join_label")
)

idx_cols = [f"{c}_idx" for c in cat_cols]
encoded_df = encoded_df.fillna(0.0, subset=idx_cols + ["label"])

print(f"Encoded dataset: {encoded_df.count():,} rows")
print(f"Label classes: {encoded_df.select('label').distinct().count()}")

## 3. Train/Test Split & Feature Pipeline

In [0]:
train_df, test_df = encoded_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  |  Test: {test_df.count():,}")

feature_cols = idx_cols + num_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features",
    handleInvalid="keep"
)

scaler = StandardScaler(inputCol="raw_features", outputCol="features")

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

results = []
print("Ready.")

## 4a. Decision Tree

In [0]:
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42,
    maxDepth=10,
    impurity="entropy"
)

dt_pipeline = Pipeline(stages=[assembler, scaler, dt])

t0 = time.time()
dt_model = dt_pipeline.fit(train_df)
dt_time = round(time.time() - t0, 2)

dt_preds = dt_model.transform(test_df)
dt_acc = evaluator_acc.evaluate(dt_preds)
dt_f1 = evaluator_f1.evaluate(dt_preds)

print(f"Decision Tree — Accuracy: {dt_acc:.4f}  |  F1: {dt_f1:.4f}  |  Time: {dt_time}s")

results.append({
    "Algorithm": "Decision Tree",
    "MLlib_Accuracy": round(dt_acc, 4),
    "MLlib_F1": round(dt_f1, 4),
    "MLlib_Train_Time": dt_time
})

del dt_model, dt_preds
gc.collect()
print("✓ DT freed")

## 4b. Random Forest

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42,
    numTrees=30,
    maxDepth=5,
    impurity="gini"
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

t0 = time.time()
rf_model = rf_pipeline.fit(train_df)
rf_time = round(time.time() - t0, 2)

rf_preds = rf_model.transform(test_df)
rf_acc = evaluator_acc.evaluate(rf_preds)
rf_f1 = evaluator_f1.evaluate(rf_preds)

print(f"Random Forest — Accuracy: {rf_acc:.4f}  |  F1: {rf_f1:.4f}  |  Time: {rf_time}s")

results.append({
    "Algorithm": "Random Forest",
    "MLlib_Accuracy": round(rf_acc, 4),
    "MLlib_F1": round(rf_f1, 4),
    "MLlib_Train_Time": rf_time
})

del rf_model, rf_preds
gc.collect()
print("✓ RF freed")

## 4c. Naive Bayes

In [0]:
nb = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    smoothing=1.0,
    modelType="multinomial"
)

nb_pipeline = Pipeline(stages=[assembler, scaler, nb])

t0 = time.time()
nb_model = nb_pipeline.fit(train_df)
nb_time = round(time.time() - t0, 2)

nb_preds = nb_model.transform(test_df)
nb_acc = evaluator_acc.evaluate(nb_preds)
nb_f1 = evaluator_f1.evaluate(nb_preds)

print(f"Naive Bayes — Accuracy: {nb_acc:.4f}  |  F1: {nb_f1:.4f}  |  Time: {nb_time}s")

results.append({
    "Algorithm": "Naive Bayes",
    "MLlib_Accuracy": round(nb_acc, 4),
    "MLlib_F1": round(nb_f1, 4),
    "MLlib_Train_Time": nb_time
})

del nb_model, nb_preds
gc.collect()
print("✓ NB freed")

## 5. Phase 2 vs Phase 3 Comparison

**IMPORTANT:** Replace the placeholder values below with your actual Phase 2 results.
Run `python3 src/models/compare_algorithms.py` locally and copy the printed metrics.

In [0]:
# ════════════════════════════════════════════════════════════════════
# Phase 2 sklearn results — replace 0.0000 with actual numbers
# Run locally: python3 src/models/compare_algorithms.py
# ════════════════════════════════════════════════════════════════════

phase2_results = {
    "Decision Tree": {"Accuracy": 0.0000, "F1": 0.0000},  # ← replace
    "Random Forest": {"Accuracy": 0.0000, "F1": 0.0000},  # ← replace
    "Naive Bayes":   {"Accuracy": 0.0000, "F1": 0.0000},  # ← replace
}

comparison_rows = []
for r in results:
    algo = r["Algorithm"]
    p2 = phase2_results[algo]
    comparison_rows.append((
        algo,
        p2["Accuracy"],
        r["MLlib_Accuracy"],
        round(r["MLlib_Accuracy"] - p2["Accuracy"], 4),
        p2["F1"],
        r["MLlib_F1"],
        round(r["MLlib_F1"] - p2["F1"], 4),
        r["MLlib_Train_Time"]
    ))

comp_df = spark.createDataFrame(
    comparison_rows,
    ["Algorithm",
     "Phase2_sklearn_Acc", "Phase3_MLlib_Acc", "Acc_Diff",
     "Phase2_sklearn_F1",  "Phase3_MLlib_F1",  "F1_Diff",
     "MLlib_Train_Time_s"]
)

print("=" * 80)
print("PHASE 2 (sklearn) vs PHASE 3 (Spark MLlib) — Crime Category Classification")
print("=" * 80)
display(comp_df)

## 6. Print Comparison Table

In [0]:
comp_pd = comp_df.toPandas()

print("\n" + "=" * 85)
print(f"{'Algorithm':<18} {'sklearn Acc':>12} {'MLlib Acc':>11} {'Δ Acc':>8} {'sklearn F1':>12} {'MLlib F1':>10} {'Δ F1':>7}")
print("-" * 85)
for _, row in comp_pd.iterrows():
    print(f"{row['Algorithm']:<18} {row['Phase2_sklearn_Acc']:>12.4f} {row['Phase3_MLlib_Acc']:>11.4f} {row['Acc_Diff']:>+8.4f} {row['Phase2_sklearn_F1']:>12.4f} {row['Phase3_MLlib_F1']:>10.4f} {row['F1_Diff']:>+7.4f}")
print("=" * 85)

## 7. Discussion

### Why might Phase 2 and Phase 3 results differ?

1. **Similar accuracy/F1 expected** — Both sklearn and MLlib implement the same algorithms 
   (CART for DT, bagged CART for RF, Bayes theorem for NB).

2. **Minor differences are normal** because of:
   - **Train/test splits:** sklearn uses stratified split; Spark's `randomSplit` does not stratify
   - **Feature encoding:** sklearn `LabelEncoder` assigns alphabetically; our `dense_rank` assigns by frequency
   - **Scaler:** sklearn's `StandardScaler` centers + scales; Spark's only scales by std (no centering)
   - **Model size constraints:** RF on serverless limited to 30 trees / depth 5 (vs sklearn's 200 trees / depth 20)

3. **Naive Bayes may differ most** — Phase 2 used `ComplementNB` (designed for imbalanced classes, 
   not available in MLlib). MLlib uses standard multinomial NB.

4. **Training time:** Spark has JVM overhead. For 62K rows, sklearn is faster. 
   Spark's advantage appears at millions of rows with distributed computing.